In [30]:
%pip install transformers==4.40.2 peft==0.11.1 datasets==2.18.0 accelerate==0.27.2 bitsandbytes==0.42.0 trl==0.8.6 sentencepiece

Note: you may need to restart the kernel to use updated packages.


In [31]:
import math
import random
import torch

from datasets import Dataset, load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import (
    LoraConfig,
    PeftModel,
    get_peft_model,
    prepare_model_for_kbit_training,
)
from trl import SFTTrainer

In [32]:
# Task: Pokemon QnA
# Domain: Pokemon Knowledge

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
DATASET_NAME = "tungdop2/pokemon"

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

In [33]:
# Load Dataset

dataset = load_dataset(DATASET_NAME)
raw_train = dataset["train"]

# Drop image columns to avoid Pillow dependency
image_like_cols = [
    col for col in raw_train.column_names if "image" in col.lower() or "img" in col.lower()
 ]
if image_like_cols:
    raw_train = raw_train.remove_columns(image_like_cols)

print(dataset)
print("Columns:", raw_train.column_names)

DatasetDict({
    train: Dataset({
        features: ['image', 'name', 'type_1', 'type_2', 'caption'],
        num_rows: 1271
    })
})
Columns: ['name', 'type_1', 'type_2', 'caption']


In [34]:
def clean_text(value):
    if value is None:
        return None
    value = str(value).strip()
    if value.lower() == "null" or value == "":
        return None
    return value

def join_types(t1, t2):
    if t1 and t2:
        return f"{t1}/{t2}"
    return t1 if t1 else (t2 if t2 else "unknown")

In [35]:
# Convert Dataset

DATASET_SYSTEM_PROMPT = (
    "You are a helpful Pokemon assistant. Answer with short, factual Pokemon information. "
    "If the answer is missing from the training data, say you do not know."
)

def build_examples(example):
    name = clean_text(example.get("name"))
    type1 = clean_text(example.get("type_1")) or clean_text(example.get("type"))
    type2 = clean_text(example.get("type_2"))
    caption = clean_text(example.get("caption"))

    if not type2:
        for key in ["type2", "secondary_type"]:
            if key in example:
                type2 = clean_text(example.get(key))
                break

    if not name:
        return []

    types = join_types(type1, type2)
    short_answer = f"{name} is a {types}-type Pokemon."

    if not caption:
        caption = short_answer

    detailed_answer = short_answer if caption == short_answer else f"{short_answer} {caption}"

    examples = [
        {"prompt": f"What type is {name}?", "completion": short_answer},
        {"prompt": f"What are {name}'s types?", "completion": short_answer},
        {"prompt": f"Describe {name}.", "completion": detailed_answer},
        {"prompt": f"Tell me about {name}.", "completion": detailed_answer},
        {"prompt": f"What kind of Pokemon is {name}?", "completion": detailed_answer},
    ]

    if type2:
        examples.append({
            "prompt": f"Is {name} a dual-type Pokemon?",
            "completion": f"Yes. {name} is a {types}-type Pokemon."
        })

    formatted = []
    for ex in examples:
        formatted.append({
            "messages": [
                {"role": "system", "content": DATASET_SYSTEM_PROMPT},
                {"role": "user", "content": ex["prompt"]},
                {"role": "assistant", "content": ex["completion"]},
            ]
        })

    return formatted

In [36]:
# Build Dataset

all_examples = []

for row in raw_train:
    all_examples.extend(build_examples(row))

random.shuffle(all_examples)

MAX_SAMPLES = min(1500, len(all_examples))
all_examples = all_examples[:MAX_SAMPLES]

print("Total examples:", len(all_examples))

Total examples: 1500


In [55]:
# Train/Validation Split (fresh datasets)

split_index = int(len(all_examples) * 0.8)

train_dataset = Dataset.from_list(all_examples[:split_index])
valid_dataset = Dataset.from_list(all_examples[split_index:])

print("Train size:", len(train_dataset))
print("Valid size:", len(valid_dataset))
print("Columns:", train_dataset.column_names)


Train size: 1200
Valid size: 300
Columns: ['messages']


In [38]:
# Tokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [39]:
# Load Model

use_cuda = torch.cuda.is_available()

if use_cuda:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto",
    )
else:
    bnb_config = None
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float32,
        device_map="cpu",
    )

model.config.use_cache = False
model.config.pad_token_id = tokenizer.pad_token_id

if use_cuda:
    model = prepare_model_for_kbit_training(model)

In [40]:
#LoRA Config

peft_config = LoraConfig(
    r=16,
    lora_alpha=8,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

trainable params: 2,162,688 || all params: 496,195,456 || trainable%: 0.4359


In [ ]:
training_args = TrainingArguments(
    output_dir="./pokemon-qwen-lora-output",
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    num_train_epochs=3,
    logging_steps=10,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    report_to="none",

    fp16=False,
    bf16=False,

    optim="adamw_torch",

    remove_unused_columns=True
)

In [ ]:
# Format and tokenize the messages
def format_and_tokenize(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False
    )
    tokenized = tokenizer(text, truncation=True, max_length=1024, padding=False)
    return tokenized

# Preprocess datasets (only if messages column exists)
if "messages" in train_dataset.column_names:
    train_dataset = train_dataset.map(format_and_tokenize, remove_columns=["messages"])
    valid_dataset = valid_dataset.map(format_and_tokenize, remove_columns=["messages"])

# Trainer - use regular Trainer since data is pre-tokenized
from transformers import Trainer, DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator, 
)

Map:   0%|          | 0/1200 [00:00<?, ? examples/s]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

In [57]:
train_result = trainer.train()
print("Training finished.")

  0%|          | 0/450 [00:00<?, ?it/s]

{'loss': 3.514, 'grad_norm': 1.58905029296875, 'learning_rate': 0.00019555555555555556, 'epoch': 0.07}
{'loss': 2.1299, 'grad_norm': 2.7622618675231934, 'learning_rate': 0.00019111111111111114, 'epoch': 0.13}
{'loss': 1.1847, 'grad_norm': 0.7311515808105469, 'learning_rate': 0.0001866666666666667, 'epoch': 0.2}
{'loss': 0.8983, 'grad_norm': 1.1392854452133179, 'learning_rate': 0.00018222222222222224, 'epoch': 0.27}
{'loss': 0.7501, 'grad_norm': 0.7473718523979187, 'learning_rate': 0.00017777777777777779, 'epoch': 0.33}
{'loss': 0.7381, 'grad_norm': 0.7676016688346863, 'learning_rate': 0.00017333333333333334, 'epoch': 0.4}
{'loss': 0.6911, 'grad_norm': 0.7489064931869507, 'learning_rate': 0.00016888888888888889, 'epoch': 0.47}
{'loss': 0.6877, 'grad_norm': 0.6444653868675232, 'learning_rate': 0.00016444444444444444, 'epoch': 0.53}
{'loss': 0.6186, 'grad_norm': 0.6730166673660278, 'learning_rate': 0.00016, 'epoch': 0.6}
{'loss': 0.6155, 'grad_norm': 1.252852201461792, 'learning_rate': 0.

  0%|          | 0/300 [00:00<?, ?it/s]

{'eval_loss': 0.5690929889678955, 'eval_runtime': 206.2277, 'eval_samples_per_second': 1.455, 'eval_steps_per_second': 1.455, 'epoch': 1.0}


c:\Users\USER\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


...

{'loss': 0.548, 'grad_norm': 0.683271586894989, 'learning_rate': 6.222222222222222e-05, 'epoch': 2.07}
{'loss': 0.4957, 'grad_norm': 0.7343940138816833, 'learning_rate': 5.7777777777777776e-05, 'epoch': 2.13}
{'loss': 0.5456, 'grad_norm': 0.6969414353370667, 'learning_rate': 5.333333333333333e-05, 'epoch': 2.2}
{'loss': 0.5102, 'grad_norm': 0.6410654187202454, 'learning_rate': 4.888888888888889e-05, 'epoch': 2.27}
{'loss': 0.491, 'grad_norm': 0.6087197065353394, 'learning_rate': 4.4444444444444447e-05, 'epoch': 2.33}
{'loss': 0.5227, 'grad_norm': 0.5737737417221069, 'learning_rate': 4e-05, 'epoch': 2.4}
{'loss': 0.5312, 'grad_norm': 0.7086345553398132, 'learning_rate': 3.555555555555556e-05, 'epoch': 2.47}
{'loss': 0.5372, 'grad_norm': 0.706165075302124, 'learning_rate': 3.111111111111111e-05, 'epoch': 2.53}
{'loss': 0.4981, 'grad_norm': 0.5219827890396118, 'learning_rate': 2.6666666666666667e-05, 'epoch': 2.6}
{'loss': 0.49, 'grad_norm': 0.7849549651145935, 'learning_rate': 2.22222222

  0%|          | 0/300 [00:00<?, ?it/s]

{'eval_loss': 0.5213760137557983, 'eval_runtime': 165.8615, 'eval_samples_per_second': 1.809, 'eval_steps_per_second': 1.809, 'epoch': 3.0}


'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 1fc8a93b-a5ad-4c4f-944d-b0dec82e4589)')' thrown while requesting HEAD https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct/resolve/main/config.json
Retrying in 1s [Retry 1/5].
c:\Users\USER\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


{'train_runtime': 11375.6069, 'train_samples_per_second': 0.316, 'train_steps_per_second': 0.04, 'train_loss': 0.6883772299024794, 'epoch': 3.0}
Training finished.


In [59]:
import math

print(trainer.state.log_history)

eval_logs = [log for log in trainer.state.log_history if "eval_loss" in log]

if eval_logs:
    final_eval = eval_logs[-1]
    final_eval_loss = final_eval["eval_loss"]
    perplexity = math.exp(final_eval_loss)

    print("Final Eval Loss:", final_eval_loss)
    print("Perplexity:", perplexity)
else:
    print("No eval_loss found in trainer logs.")

[{'loss': 3.514, 'grad_norm': 1.58905029296875, 'learning_rate': 0.00019555555555555556, 'epoch': 0.06666666666666667, 'step': 10}, {'loss': 2.1299, 'grad_norm': 2.7622618675231934, 'learning_rate': 0.00019111111111111114, 'epoch': 0.13333333333333333, 'step': 20}, {'loss': 1.1847, 'grad_norm': 0.7311515808105469, 'learning_rate': 0.0001866666666666667, 'epoch': 0.2, 'step': 30}, {'loss': 0.8983, 'grad_norm': 1.1392854452133179, 'learning_rate': 0.00018222222222222224, 'epoch': 0.26666666666666666, 'step': 40}, {'loss': 0.7501, 'grad_norm': 0.7473718523979187, 'learning_rate': 0.00017777777777777779, 'epoch': 0.3333333333333333, 'step': 50}, {'loss': 0.7381, 'grad_norm': 0.7676016688346863, 'learning_rate': 0.00017333333333333334, 'epoch': 0.4, 'step': 60}, {'loss': 0.6911, 'grad_norm': 0.7489064931869507, 'learning_rate': 0.00016888888888888889, 'epoch': 0.4666666666666667, 'step': 70}, {'loss': 0.6877, 'grad_norm': 0.6444653868675232, 'learning_rate': 0.00016444444444444444, 'epoch':

In [60]:
trainer.model.save_pretrained("./pokemon-qwen-lora-final")
tokenizer.save_pretrained("./pokemon-qwen-lora-final")

print("Model saved!")

c:\Users\USER\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Model saved!


In [61]:
INFERENCE_SYSTEM_PROMPT = (
    "You are a helpful Pokemon assistant. Answer clearly using the Pokemon data learned in this notebook. "
    "If you are not sure, say you do not know instead of guessing."
)
MODEL_SAVE_PATH = "./pokemon-qwen-lora-final"
GENERATION_CONFIG = {
    "max_new_tokens": 96,
    "do_sample": False,
    "repetition_penalty": 1.05,
}

def ensure_inference_objects():
    global model, tokenizer

    if "model" in globals() and "tokenizer" in globals() and model is not None and tokenizer is not None:
        model.eval()
        return

    tokenizer = AutoTokenizer.from_pretrained(MODEL_SAVE_PATH)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token_id = tokenizer.eos_token_id

    quantization_config = bnb_config if "bnb_config" in globals() and torch.cuda.is_available() else None
    device_map = "auto" if torch.cuda.is_available() else None
    torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=quantization_config,
        device_map=device_map,
        torch_dtype=torch_dtype,
    )

    if not torch.cuda.is_available():
        base_model = base_model.to("cpu")

    base_model.config.pad_token_id = tokenizer.pad_token_id
    model = PeftModel.from_pretrained(base_model, MODEL_SAVE_PATH)
    model.eval()

def ask_model(prompt, history=None):
    ensure_inference_objects()

    if not prompt or not prompt.strip():
        return "Please ask a Pokemon question."

    history = history or []
    messages = [{"role": "system", "content": INFERENCE_SYSTEM_PROMPT}]
    messages.extend(history)
    messages.append({"role": "user", "content": prompt.strip()})

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(text, return_tensors="pt")
    target_device = next(model.parameters()).device
    inputs = {key: value.to(target_device) for key, value in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            **GENERATION_CONFIG,
        )

    prompt_length = inputs["input_ids"].shape[-1]
    new_tokens = outputs[0][prompt_length:]
    answer = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    return answer if answer else "I do not know."

In [ ]:
prompts = [
    "What type is Pikachu?",
    "Can you describe Bulbasaur?",
    "Tell me about Charizard.",
    "Is Gyarados a dual-type Pokemon?",
]

for p in prompts:
    print("PROMPT:", p)
    print("RESPONSE:", ask_model(p))
    print("-" * 50)

PROMPT: What type is Pikachu?


c:\Users\USER\AppData\Local\Programs\Python\Python311\Lib\site-packages\transformers\generation\configuration_utils.py:492: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
c:\Users\USER\AppData\Local\Programs\Python\Python311\Lib\site-packages\transformers\generation\configuration_utils.py:497: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
c:\Users\USER\AppData\Local\Programs\Python\Python311\Lib\site-packages\transformers\generation\configuration_utils.py:509: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


RESPONSE: Pikachu is a electric-type Pokemon.
--------------------------------------------------
PROMPT: Can you describe Bulbasaur?
RESPONSE: Bulbasaur is a grass-type Pokemon. A green, round creature with a long tail and a pair of large, sharp claws. It has a cute and innocent appearance, resembling a small, furry creature.
--------------------------------------------------
PROMPT: Tell me about Charizard.
RESPONSE: Charizard is a fire-type Pokemon. A large, red and yellow creature with a long tail, resembling a dragon or a dragonfly. It has a unique appearance and is likely a popular character in various video games and media.
--------------------------------------------------
PROMPT: Is Gyarados a dual-type Pokemon?
RESPONSE: Yes, Gyarados is a water/ground-type Pokemon.
--------------------------------------------------


In [63]:
def chat(max_history_messages=8):
    print("Pokemon Assistant (type 'clear' to reset or 'exit' to stop)\n")

    history = []

    while True:
        user_input = input("You: ").strip()

        if not user_input:
            continue

        lowered = user_input.lower()

        if lowered in ["exit", "quit"]:
            print("Goodbye!")
            break

        if lowered == "clear":
            history = []
            print("Conversation cleared.\n")
            continue

        answer = ask_model(user_input, history=history)

        history.append({"role": "user", "content": user_input})
        history.append({"role": "assistant", "content": answer})
        history = history[-max_history_messages:]

        print(f"\nBot: {answer}\n")

chat()

Pokemon Assistant (type 'clear' to reset or 'exit' to stop)


Bot: treecko is a tree-type Pokemon.


Bot: treecko is a tree-type Pokemon.


Bot: treecko is a green and white creature with a long tail, resembling a dragon or a snake. It has a unique appearance and is likely to be a popular choice for a Pokemon character.


Bot: sceptile is a fire-type Pokemon.


Bot: mudkip is a water-type Pokemon.


Bot: escape is a water-type Pokemon.

Goodbye!
